---
# Topic 4: Exception Handling

An **exception** is an error that occurs at runtime. Without exception handling, any error crashes your entire program. In ML pipelines that run for hours, a single bad data sample could stop everything. Exception handling is how you build **robust, production-grade code**.

---

## 4.1 The Exception Hierarchy

### 📊 [VISUAL] Python Exception Hierarchy

```
BaseException
├── SystemExit              <- sys.exit() raises this
├── KeyboardInterrupt       <- Ctrl+C raises this
└── Exception               <- ALL normal errors inherit from here
     ├── ValueError           <- wrong value type (int('abc'))
     ├── TypeError            <- wrong type operation ('a' + 1)
     ├── AttributeError       <- obj has no such attribute
     ├── KeyError             <- dict key doesn't exist
     ├── IndexError           <- list index out of range
     ├── FileNotFoundError    <- file doesn't exist
     ├── ZeroDivisionError    <- division by zero
     ├── ImportError          <- module not found
     ├── StopIteration        <- iterator exhausted
     ├── RuntimeError         <- generic runtime error
     ├── OverflowError        <- number too large
     └── MemoryError          <- out of RAM (common in ML!)
```

---

## 4.2 try / except / else / finally — Complete Structure

### 📊 [VISUAL] Execution Flow

```
try:                        <- attempt the risky code
    risky_code()
except SpecificError as e:  <- handle one specific type
    handle_it(e)
except (ErrorA, ErrorB):    <- handle multiple types together
    handle_both()
except Exception as e:      <- catch any remaining exception
    log(e)
else:                       <- runs ONLY if NO exception was raised
    success_code()
finally:                    <- ALWAYS runs, exception or not (cleanup)
    cleanup()

EXECUTION PATHS:
  No error:      try -> else -> finally
  Error caught:  try -> except -> finally
  Error uncaught: try -> finally -> propagates up
```

In [ ]:
import csv
from pathlib import Path

def load_dataset(filepath):
    """Load CSV dataset with full exception handling."""
    data = []
    try:
        if not Path(filepath).exists():
            raise FileNotFoundError(f'Dataset not found: {filepath}')

        with open(filepath, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row_num, row in enumerate(reader, start=2):  # 1=header
                try:
                    record = {
                        'name':  row['name'].strip(),
                        'age':   int(row['age']),
                        'gpa':   float(row['gpa'])
                    }
                    data.append(record)
                except (ValueError, KeyError) as row_err:
                    print(f'  Warning: Skipping row {row_num} — {row_err}')
                    continue    # skip bad row, keep loading

    except FileNotFoundError as e:
        print(f'Error: {e}')
        return None

    except PermissionError:
        print(f'Error: No permission to read {filepath}')
        return None

    except Exception as e:
        print(f'Unexpected error: {type(e).__name__}: {e}')
        raise   # re-raise unexpected errors — never swallow silently

    else:
        print(f'Dataset loaded: {len(data)} records')

    finally:
        print('load_dataset() finished.')

    return data


# Test with existing file
result = load_dataset('students.csv')
if result:
    for r in result:
        print(f"  {r}")

print()

# Test with missing file
result2 = load_dataset('nonexistent.csv')
print(f"Result for missing file: {result2}")

## 4.3 Custom Exceptions

Custom exceptions create domain-specific error types with meaningful names. You can tell immediately from the exception name what went wrong.

In [ ]:
# ── Custom exception hierarchy ────────────────────────────────────────────────
class MLError(Exception):
    """Base exception for all ML-related errors."""
    pass

class UntrainedModelError(MLError):
    """Raised when predict() is called on an untrained model."""
    def __init__(self, model_name):
        self.model_name = model_name
        super().__init__(f"Model '{model_name}' must be trained before prediction")

class DataShapeError(MLError):
    """Raised when input data has incorrect shape."""
    def __init__(self, expected, actual):
        super().__init__(f'Expected shape {expected}, got {actual}')
        self.expected = expected
        self.actual   = actual

class InvalidHyperparameterError(MLError):
    """Raised when a hyperparameter value is out of valid range."""
    pass


# ── Using custom exceptions ───────────────────────────────────────────────────
class SimpleModel:
    def __init__(self, name, lr):
        if not (0 < lr < 1):
            raise InvalidHyperparameterError(
                f'Learning rate must be in (0,1), got {lr}'
            )
        self.name        = name
        self.lr          = lr
        self.is_trained  = False
        self._n_features = None

    def fit(self, X, y):
        self._n_features = len(X[0])
        self.is_trained  = True

    def predict(self, X):
        if not self.is_trained:
            raise UntrainedModelError(self.name)
        if len(X[0]) != self._n_features:
            raise DataShapeError(expected=self._n_features, actual=len(X[0]))
        return [1 if x[0] > 0 else 0 for x in X]


# ── Test all exception paths ──────────────────────────────────────────────────
# Path 1: untrained model
model = SimpleModel('MyModel', 0.01)
try:
    predictions = model.predict([[1, 2, 3]])
except UntrainedModelError as e:
    print(f'Caught UntrainedModelError: {e}')

# Path 2: wrong shape
model.fit([[1,2,3],[4,5,6]], [0, 1])
try:
    predictions = model.predict([[1, 2]])   # expects 3 features
except DataShapeError as e:
    print(f'Caught DataShapeError: expected {e.expected}, got {e.actual}')

# Path 3: bad hyperparameter
try:
    bad_model = SimpleModel('Bad', lr=5.0)
except InvalidHyperparameterError as e:
    print(f'Caught InvalidHyperparameterError: {e}')

# Path 4: correct usage
predictions = model.predict([[1, 2, 3], [-1, 0, 0]])
print(f'Predictions: {predictions}')

## 4.4 The `raise` Statement and Re-Raising

In [ ]:
def validate_age(age):
    if not isinstance(age, int):
        raise TypeError(f'Age must be int, got {type(age).__name__}')
    if age < 0 or age > 150:
        raise ValueError(f'Age must be between 0 and 150, got {age}')
    return age


def process_record(record):
    try:
        age = validate_age(record.get('age'))
    except ValueError as e:
        # Add context and re-raise
        raise ValueError(f"Invalid record {record}: {e}") from e
    return age


# assert — lightweight validation (disabled with python -O)
def train(X, y):
    assert len(X) == len(y), f'X and y must have same length: {len(X)} vs {len(y)}'
    assert len(X) > 0, 'Training data cannot be empty'
    print("Validation passed. Training...")

# Tests
try:
    validate_age("twenty")
except TypeError as e:
    print(f"TypeError: {e}")

try:
    validate_age(200)
except ValueError as e:
    print(f"ValueError: {e}")

try:
    process_record({'age': -5, 'name': 'Alice'})
except ValueError as e:
    print(f"Re-raised: {e}")

train([[1,2],[3,4]], [0,1])

## 4.5 Context Managers — `with` Statement Deep Dive

The `with` statement is Python's context manager protocol. It **guarantees** that setup and teardown code always runs — whether the block succeeds or raises an exception.

> 🔵 **[AI/ML]** PyTorch uses context managers everywhere:
> ```python
> with torch.no_grad():          # disables gradient computation during inference
>     predictions = model(X)    # faster, less memory
>
> with torch.cuda.amp.autocast():  # automatic mixed precision (faster training)
>     loss = model(X)
> ```
> Understanding `__enter__`/`__exit__` helps you understand WHAT these do and WHY they are structured this way.

In [ ]:
import time
from contextlib import contextmanager

# ── Context manager via class ─────────────────────────────────────────────────
class Timer:
    """Context manager that times a code block."""

    def __enter__(self):
        self.start = time.perf_counter()
        return self                  # 'as' variable = what __enter__ returns

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.elapsed = time.perf_counter() - self.start
        print(f'Elapsed: {self.elapsed:.4f}s')
        return False   # False = do NOT suppress exceptions


with Timer() as t:
    total = sum(range(1_000_000))
print(f"Sum: {total}")


# ── Context manager via @contextmanager decorator ─────────────────────────────
@contextmanager
def managed_file(path, mode='r'):
    """Open file, yield it, always close it."""
    f = open(path, mode)
    try:
        yield f
    finally:
        f.close()
        print(f'Closed: {path}')


with managed_file('notes.txt', 'w') as f:
    f.write('Context managers are clean.')

with managed_file('notes.txt', 'r') as f:
    print(f.read())

## 4.6 Common Exception Handling Mistakes

```
❌ Bare except: — catches EVERYTHING including KeyboardInterrupt (Ctrl+C)
   Never write:  except:
   Always write: except Exception:  at minimum

❌ Silently swallowing exceptions:
   except Exception: pass
   This hides bugs. At minimum: except Exception as e: print(e)

❌ Catching too broadly then treating all errors the same way.
   Catch specific exceptions first. Put broad catches last.

❌ Using assert for input validation in production code.
   Assertions are disabled with python -O.
   Use explicit if/raise instead for production validation.
```

---

## ✏️ Exercises — Exception Handling

**[EXERCISE 7 — Medium]** Write `safe_divide(a, b)` that handles `ZeroDivisionError` and `TypeError` with clear messages. Then write `safe_json_load(filepath)` that handles `FileNotFoundError`, `PermissionError`, and `json.JSONDecodeError`.

**[EXERCISE 8 — Advanced]** Create a `DataPipeline` class with: `load(filepath)` loading a CSV, `validate()` checking for missing/invalid values, `transform(fn)` applying a function to each row, `save(filepath)` writing results to JSON. All methods should handle exceptions and return `self` (method chaining).